In [ ]:
pip install scikits.bootstrap

In [ ]:
# prompt: gostaria de carregar uma base que se encontra no meu google drive

from google.colab import drive
drive.mount('/content/drive')

# Substitua 'caminho/para/seu/arquivo.csv' pelo caminho real do seu arquivo no Google Drive
# Exemplo: '/content/drive/My Drive/NomeDaSuaPasta/NomeDoSeuArquivo.csv'
caminho_arquivo = '/content/drive/MyDrive/Mestrado/Dissertaçâo /df_final.csv'

import pandas as pd

try:
  #df = pd.read_csv(caminho_arquivo)
  df = pd.read_csv('/content/df_final_brazilian.csv', sep=',')
  print("Base de dados carregada com sucesso!")
  print(df.head()) # Mostra as primeiras linhas do dataframe para verificar
except FileNotFoundError:
  print(f"Erro: O arquivo '{caminho_arquivo}' não foi encontrado.")
except Exception as e:
  print(f"Ocorreu um erro ao carregar o arquivo: {e}")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Base de dados carregada com sucesso!
  dominant_gender   age dominant_emotion    dominant_race     asian    indian  \
0           Woman  28.0            happy  latino hispanic  0.834433  1.896242   
1           Woman  30.0            happy  latino hispanic  0.717828  2.432063   
2           Woman  27.0            happy            white  2.534839  3.590405   
3           Woman  29.0          neutral            white  0.232616  0.804510   
4           Woman  27.0            happy  latino hispanic  3.099153  8.328156   

      black      white  middle_eastern  latino_hispanic  empresa  indice  \
0  0.627464  39.109085        6.597933        50.934845  imagine       1   
1  0.470095  35.523758       13.957540        46.898716  imagine       2   
2  1.389204  39.939346       15.899272        36.646930  imagine       3   
3  0.078757  45.731080       10.570088     

In [ ]:
df

,dominant_gender,age,dominant_emotion,dominant_race,asian,indian,black,white,middle_eastern,latino_hispanic,empresa,indice,prompt
0,Woman,28.0,happy,latino hispanic,0.834433,1.896242,0.627464,39.109085,6.597933,50.934845,imagine,1,generate a brazilian
1,Woman,30.0,happy,latino hispanic,0.717828,2.432063,0.470095,35.523758,13.957540,46.898716,imagine,2,generate a brazilian
2,Woman,27.0,happy,white,2.534839,3.590405,1.389204,39.939346,15.899272,36.646930,imagine,3,generate a brazilian
3,Woman,29.0,neutral,white,0.232616,0.804510,0.078757,45.731080,10.570088,42.582954,imagine,4,generate a brazilian
4,Woman,27.0,happy,latino hispanic,3.099153,8.328156,1.216155,13.513340,6.261908,67.581300,imagine,5,generate a brazilian
...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,Man,27.0,neutral,indian,1.721634,45.924940,41.905552,1.405210,1.113947,7.928717,google,26,generate a brazilian
118,Man,24.0,sad,white,0.000969,0.000303,0.000021,99.003365,0.320864,0.674481,google,27,generate a brazilian
119,Man,29.0,fear,asian,30.518818,11.299201,28.944866,9.312797,5.373815,14.550512,google,28,generate a brazilian
120,Man,25.0,fear,white,0.819108,0.140959,0.224610,86.890340,4.202848,7.722139,google,29,generate a brazilian


In [ ]:
# prompt: gostaria de filtrar apenas os dados da empresa google

# Filtran o DataFrame para incluir apenas dados da empresa "Google"
df_google = df[df['empresa'] == 'google'].copy()
df_copilot = df[df['empresa'] == 'copilot'].copy()
df_imagine = df[df['empresa'] == 'imagine'].copy()
df_adobe = df[df['empresa'] == 'adobe'].copy()


In [ ]:
# prompt: quero através do método de monte carlo estimar a probabilidade de cada classe ocorrer dado a minha coluna categórica de interesse, com 90% de precisão e com 10000 reamostragens, gostaria do intervalo de confiança da probabilidade também. Gostaria que utiliza-se a distribuição normal, a taxa de erro é de 5% e a probabilidade de referência é de 49% man e 51% woman; a solução deverá aparecer em formato de uma função

import numpy as np
import pandas as pd
from scipy.stats import norm
from scikits.bootstrap import bootstrap as bootstrap_skb

def monte_carlo_categorical_probability(dataframe, column_name, n_simulations=10000, reference_probabilities=None, erro_aceito = 0.1):
    """
    Estimates the probability of each class in a categorical column using the Monte Carlo method.

    Args:
        dataframe (pd.DataFrame): The input DataFrame.
        column_name (str): The name of the categorical column.
        confidence_level (float): The desired confidence level for the interval (e.g., 0.90 for 90%).
        n_simulations (int): The number of Monte Carlo simulations (resamplings).
        reference_probabilities (dict, optional): A dictionary of reference probabilities for each class.
                                                 Keys are class names, values are probabilities.

    Returns:
        dict: A dictionary where keys are class names and values are dictionaries
              containing 'estimated_probability', 'confidence_interval', and 'matches_reference'
              (if reference_probabilities are provided).
    """
    if column_name not in dataframe.columns:
        return {"error": f"Column '{column_name}' not found in the DataFrame."}

    class_counts = dataframe[column_name].value_counts()
    total_samples = len(dataframe)
    class_probabilities = class_counts / total_samples

    results = {}

    for class_name, initial_prob in class_probabilities.items():
        simulated_probabilities = []
        for _ in range(n_simulations):
            # Resample with replacement
            resampled_data = dataframe[column_name].sample(n=total_samples, replace=True)
            simulated_prob = (resampled_data == class_name).sum() / total_samples
            simulated_probabilities.append(simulated_prob)

        # Calculate estimated probability (mean of simulations)
        #estimated_probability = np.mean(simulated_probabilities)
        simulated_probabilities = np.array(simulated_probabilities) # Convert list to NumPy array
        estimated_probability = np.sum(sum((simulated_probabilities >= reference_probs[class_name] - erro_aceito) & (simulated_probabilities <= reference_probs[class_name] + erro_aceito)))/n_simulations



        # Corrected: Calculate bounds based on the reference probability for the current class
        if reference_probabilities and class_name in reference_probabilities:
            ref_prob = reference_probabilities[class_name]
            lower_bound = ref_prob - erro_aceito
            upper_bound = ref_prob + erro_aceito
        else:

            lower_bound = None
            upper_bound = None


        confidence_interval = (lower_bound, upper_bound)

        class_result = {
            "estimated_probability": estimated_probability,
            "confidence_interval": confidence_interval
        }

        # Check if the estimated probability matches the reference probability within the margin of error
        if reference_probabilities and class_name in reference_probabilities:
            ref_prob = reference_probabilities[class_name]
            class_result["matches_reference"] = estimated_probability > 0.9

        results[class_name] = class_result

    return results





In [ ]:
# Example usage with your loaded DataFrame 'df'
# Assuming the column of interest is 'Gender' (replace with your actual column name)
# Assuming reference probabilities are for 'man' and 'woman'

#column_of_interest = 'dominant_gender' # Replace with the actual categorical column name
column_of_interest = 'dominant_race' # Replace with the actual categorical column name

# referecia tirado do IBGE
#reference_probs = {'Man': 0.49, 'Woman': 0.51} # Replace with your actual class names and reference probabilities
reference_probs = {'white': 0.43, 'black': 0.55, 'latino hispanic': 0.00, 'middle eastern': 0.00, 'indian': 0.00, 'asian': 0.01} # Replace with your actual class names and reference probabilities

base = df_imagine
taxa_erro = 0.05

# Ensure the column exists in the DataFrame before calling the function
if column_of_interest in df.columns:
    probability_estimates = monte_carlo_categorical_probability(
        base,
        column_of_interest,
        n_simulations=10000,
        reference_probabilities=reference_probs,
        erro_aceito = taxa_erro
    )

    print("\nMonte Carlo Probability Estimates:")
    for class_name, data in probability_estimates.items():
        print(f"Class: {class_name}")
        print(f"  Probabilidade Estimada: {data['estimated_probability']:.4f}")
        print(f"  Intervalo de confiança +- ({taxa_erro * 100}%): ({data['confidence_interval'][0]:.4f}, {data['confidence_interval'][1]:4f})")

                # --------- Cálculo do intervalo de confiança BCa ---------
        # Cria vetor booleano para a classe atual
        class_vector = (base[column_of_interest] == class_name).astype(int).values

        # Usa scikits.bootstrap para estimar intervalo BCa da média (proporção)
        try:
            bca_ci = bootstrap_skb.ci(
                class_vector,
                statfunction=np.mean,
                alpha=2 * taxa_erro,
                method='bca',
                n_samples=10000
            )
            print(f"  Intervalo de confiança BCa: ({bca_ci[0]:.4f}, {bca_ci[1]:.4f})")
        except Exception as e:
            print(f"  [Erro ao calcular intervalo BCa: {e}]")


        if 'matches_reference' in data:
            print(f"  A coleção de imagem não possui viés?: {data['matches_reference']}")
            # Corrected indentation and comparison operator
            if data['matches_reference'] == False and column_of_interest == 'dominant_race':

                class_counts = base[column_of_interest].value_counts()
                total_samples = len(base)
                class_probabilities = class_counts / total_samples
#                min_count_class_percent = class_probabilities.idxmin()
                min_count_percentage_value = class_probabilities.min() * 100
                total_percentage_sum = class_probabilities.sum() * 100
                sum_excluding_min = (total_percentage_sum - min_count_percentage_value)

                # Current samples of minority class
                current_minority_samples = total_samples * (min_count_percentage_value / 100)

                # Current samples of all other classes
                current_other_samples = total_samples - current_minority_samples
                new_total_samples = current_other_samples / (1 - reference_probs[class_name])

                adicionar_total_samples = new_total_samples - total_samples

                # And the number of samples of the minority class to add is:
                adicionar_minority_class_samples = (new_total_samples * reference_probs[class_name]) - current_minority_samples

                # Corrected print statement
#                print(f"  Gere aproximadamente {adicionar_minority_class_samples:.0f} amostras adicionais da classe '{min_count_class_percent}'.")
                print(f"  Gere aproximadamente {adicionar_minority_class_samples:.0f} amostras adicionais da classe '{class_name}'.")

            if data['matches_reference'] == False and column_of_interest == 'dominant_gender':

                class_counts = base[column_of_interest].value_counts()
                total_samples = len(base)
                class_probabilities = class_counts / total_samples
                min_count_class_percent = class_probabilities.idxmin()
                min_count_percentage_value = class_probabilities.min() * 100
                total_percentage_sum = class_probabilities.sum() * 100
                sum_excluding_min = (total_percentage_sum - min_count_percentage_value)

                # Current samples of minority class
                current_minority_samples = total_samples * (min_count_percentage_value / 100)

                # Current samples of all other classes
                current_other_samples = total_samples - current_minority_samples
                new_total_samples = current_other_samples / (1 - reference_probs[min_count_class_percent])

                adicionar_total_samples = new_total_samples - total_samples

                # And the number of samples of the minority class to add is:
                adicionar_minority_class_samples = (new_total_samples * reference_probs[min_count_class_percent]) - current_minority_samples

                # Corrected print statement
                print(f"  Gere aproximadamente {adicionar_minority_class_samples:.0f} amostras adicionais da classe '{min_count_class_percent}'.")



        print("-" * 20)
else:
    print(f"Error: The column '{column_of_interest}' does not exist in the DataFrame.")

NameError: name 'df_imagine' is not defined

In [ ]:
class_vector = (base[column_of_interest] == class_name).astype(int).values

In [ ]:
bca_ci = bootstrap_skb.ci(
                class_vector,
                statfunction=np.mean,
                alpha=2 * taxa_erro,
                method='bca',
                n_samples=10000)

In [ ]:
bca_ci

array([0.        , 0.03333333])

In [ ]:
class_vector

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0])

In [ ]:
base

,dominant_gender,age,dominant_emotion,dominant_race,asian,indian,black,white,middle_eastern,latino_hispanic,empresa,indice,prompt
0,Woman,28.0,happy,latino hispanic,0.834433,1.896242,0.627464,39.109085,6.597933,50.934845,imagine,1,generate a brazilian
1,Woman,30.0,happy,latino hispanic,0.717828,2.432063,0.470095,35.523758,13.957540,46.898716,imagine,2,generate a brazilian
2,Woman,27.0,happy,white,2.534839,3.590405,1.389204,39.939346,15.899272,36.646930,imagine,3,generate a brazilian
3,Woman,29.0,neutral,white,0.232616,0.804510,0.078757,45.731080,10.570088,42.582954,imagine,4,generate a brazilian
4,Woman,27.0,happy,latino hispanic,3.099153,8.328156,1.216155,13.513340,6.261908,67.581300,imagine,5,generate a brazilian
5,Woman,29.0,neutral,latino hispanic,4.932726,3.944383,0.677499,17.503418,2.776935,70.165040,imagine,6,generate a brazilian
6,Woman,29.0,happy,white,0.047801,0.177969,0.009195,74.160730,6.767007,18.837305,imagine,7,generate a brazilian
7,Woman,29.0,happy,white,1.469593,1.539568,0.244750,52.460953,8.521741,35.763397,imagine,8,generate a brazilian
8,Woman,30.0,happy,latino hispanic,0.268298,1.433050,0.258494,15.470780,2.423096,80.146286,imagine,9,generate a brazilian
9,Woman,28.0,neutral,latino hispanic,5.650465,5.875091,1.258920,19.897915,4.993639,62.323975,imagine,10,generate a brazilian


In [ ]:
print('Percentual da variável dominant_gender no df_adobe:')
display(base['dominant_gender'].value_counts(normalize=True) * 100)

print('\nPercentual da variável dominant_race no df_adobe:')
display(base['dominant_race'].value_counts(normalize=True) * 100)

Percentual da variável dominant_gender no df_adobe:


,proportion
dominant_gender,
Woman,100.0



Percentual da variável dominant_race no df_adobe:


,proportion
dominant_race,
latino hispanic,56.666667
white,43.333333


In [ ]:
dataframe = df_adobe
column_name = 'dominant_race'
n_simulations=10000
#reference_probs = {'Man': 0.49, 'Woman': 0.51}
reference_probs = {'white': 0.435, 'black': 0.555, 'latino hispanic': 0.00, 'middle eastern': 0.00, 'indian': 0.00, 'asian': 0.004}

In [ ]:


    class_counts = dataframe[column_name].value_counts()
    total_samples = len(dataframe)
    class_probabilities = class_counts / total_samples

    results = {}

In [ ]:
        erro_aceito = 0.1
        simulated_probabilities = []
        for _ in range(10000):
            # Resample with replacement
            resampled_data = dataframe[column_name].sample(n=total_samples, replace=True)
            simulated_prob = (resampled_data == class_name).sum() / total_samples
            simulated_probabilities.append(simulated_prob)



In [ ]:
simulated_probabilities

[np.float64(0.0),
 np.float64(0.0),
 np.float64(0.03333333333333333),
 np.float64(0.1),
 np.float64(0.03333333333333333),
 np.float64(0.03333333333333333),
 np.float64(0.03333333333333333),
 np.float64(0.1),
 np.float64(0.13333333333333333),
 np.float64(0.0),
 np.float64(0.03333333333333333),
 np.float64(0.03333333333333333),
 np.float64(0.1),
 np.float64(0.0),
 np.float64(0.06666666666666667),
 np.float64(0.03333333333333333),
 np.float64(0.03333333333333333),
 np.float64(0.03333333333333333),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.03333333333333333),
 np.float64(0.03333333333333333),
 np.float64(0.06666666666666667),
 np.float64(0.0),
 np.float64(0.06666666666666667),
 np.float64(0.03333333333333333),
 np.float64(0.03333333333333333),
 np.float64(0.0),
 np.float64(0.0),
 np.float64(0.03333333333333333),
 np.float64(0.06666666666666667),
 np.float64(0.1),
 np.float64(0.03333333333333333),
 np.float64(0.0),
 np.float64(0.03333333333333333),
 np.float64(0.0),
 np.float64(0.0)

In [ ]:
        simulated_probabilities = np.array(simulated_probabilities) # Convert list to NumPy array
        estimated_probability = np.sum(sum((simulated_probabilities >= reference_probs[class_name] - erro_aceito) & (simulated_probabilities <= reference_probs[class_name] + erro_aceito)))/n_simulations


In [ ]:
estimated_probability

np.float64(0.9833)

In [ ]:
reference_probs

{'white': 0.435,
 'black': 0.555,
 'latino hispanic': 0.0,
 'middle eastern': 0.0,
 'indian': 0.0,
 'asian': 0.004}

In [ ]:
class_name

'asian'

In [ ]:
sum((simulated_probabilities >= reference_probs[class_name] - erro_aceito) & (simulated_probabilities <= reference_probs[class_name] + erro_aceito))

np.int64(4112)

In [ ]:
        reference_probabilities = reference_probs
        lower_bound = estimated_probability - erro_aceito
        upper_bound = estimated_probability + erro_aceito

        confidence_interval = (lower_bound, upper_bound)

        class_result = {
            "estimated_probability": estimated_probability,
            "confidence_interval": confidence_interval
        }

        # Check if the estimated probability matches the reference probability within the margin of error
        if reference_probabilities and class_name in reference_probabilities:
            ref_prob = reference_probabilities[class_name]
            class_result["matches_reference"] = estimated_probability > 0.5

        results[class_name] = class_result

In [ ]:
results

{'asian': {'estimated_probability': np.float64(0.9833),
  'confidence_interval': (np.float64(0.8833), np.float64(1.0833)),
  'matches_reference': np.True_}}

In [ ]:
        # Corrected: Calculate bounds based on the reference probability for the current class
        if reference_probabilities and class_name in reference_probabilities:
            ref_prob = reference_probabilities[class_name]
            lower_bound = ref_prob - erro_aceito
            upper_bound = ref_prob + erro_aceito
        else:
            # If no reference probability for this class, maybe calculate confidence interval differently
            # Or perhaps the intention is only to calculate bounds when reference_probabilities is provided for the class.
            # For now, set bounds to None or use the estimated probability if no reference is available.
            # Using estimated probability to calculate bounds if no reference is given.
            # This part needs clarification on desired behavior if reference_probabilities is missing for a class.
            # Based on the original code trying to use reference_probabilities for bounds, we'll assume bounds are only relevant with reference.
            lower_bound = None
            upper_bound = None
            # Alternatively, you could use the estimated probability and a standard method for confidence intervals (e.g., using std_error and a z-score)
            # But the original code implies bounds based on reference probability.


In [ ]:
upper_bound

0.10400000000000001

In [ ]:
# prompt: quero dentro de class_probabilities saber qual é aquele que possui o menor count em percentual

# Find the class with the minimum percentage count
min_count_class_percent = class_probabilities.idxmin()
min_count_percentage_value = class_probabilities.min() * 100
sum_excluding_min = (total_percentage_sum - min_count_percentage_value)
(total_samples - (min_count_percentage_value * total_samples)) / (1 - reference_probs[min_count_class_percent])


((min_count_percentage_value * total_samples) * reference_probs[min_count_class_percent])/min_count_percentage_value

print(f"\nClass with the lowest percentage count: '{min_count_class_percent}' with {min_count_percentage_value:.2f}%")




Class with the lowest percentage count: 'asian' with 3.33%


In [ ]:
((total_samples - (min_count_percentage_value/100 * total_samples))/ (1 - reference_probs[min_count_class_percent] )) - (total_samples - (min_count_percentage_value/100 * total_samples)) - (total_samples * min_count_percentage_value/100)

-0.8835341365461851

In [ ]:
((total_samples - (min_count_percentage_value/100 * total_samples))/ (1 - reference_probs[min_count_class_percent] )) - (total_samples - (min_count_percentage_value/100 * total_samples))


0.11646586345381493

In [ ]:
(total_samples * min_count_percentage_value/100)

1.0

In [ ]:
# prompt: agora quero o percentual das somas das classes do class_probabilities retirando a classe com o menor percentual

# Calculate the total sum of percentages
total_percentage_sum = class_probabilities.sum() * 100

# Calculate the sum of percentages excluding the lowest class
sum_excluding_min = (total_percentage_sum - min_count_percentage_value)

# Calculate the percentage of this sum relative to the total original sum
percentage_of_sum_excluding_min = (sum_excluding_min / total_percentage_sum) * 100

print(f"Total percentage sum of all classes: {total_percentage_sum:.2f}%")
print(f"Sum of percentages excluding the lowest class ('{min_count_class_percent}'): {sum_excluding_min:.2f}%")
print(f"Percentage of the sum excluding the lowest class relative to the total sum: {percentage_of_sum_excluding_min:.2f}%")

Total percentage sum of all classes: 100.00%
Sum of percentages excluding the lowest class ('asian'): 96.67%
Percentage of the sum excluding the lowest class relative to the total sum: 96.67%


In [ ]:
reference_probs[min_count_class_percent]

0.51

In [ ]:
total_samples

30

In [ ]:
df_google['dominant_race'].value_counts()

,count
dominant_race,
black,20
white,4
latino hispanic,4
indian,2


In [ ]:
df_google['dominant_gender'].value_counts()

,count
dominant_gender,
Man,25
Woman,5


In [ ]:
# prompt: quantos man e woman tem na base df_google

man_count = df_copilot[df_copilot['dominant_gender'] == 'Man'].shape[0]
woman_count = df_copilot[df_copilot['dominant_gender'] == 'Woman'].shape[0]

print(f"Número de 'Man' na base df_copilot: {man_count}")
print(f"Número de 'Woman' na base df_copilot: {woman_count}")


Número de 'Man' na base df_copilot: 21
Número de 'Woman' na base df_copilot: 9


In [ ]:
resultados_monte_carlo

NameError: name 'resultados_monte_carlo' is not defined